# CV Skill Parser — High-Performance Rebuild
## What changed and why
| Problem | Fix |
|---|---|
| `spacy.blank` — no language knowledge | `en_core_web_lg` — pre-trained vectors |
| `alignment_mode="contract"` — dropped 19% spans | `"expand"` — recovers them |
| Noisy labels (`"com"`, `"marital status"`, `"is"`) | Filter step removes noise |
| `patience=1600`, no epoch cap — 5hr waste | `patience=5`, `max_epochs=30` |
| `width=96` shallow encoder | `width=256`, `depth=4` |
| Surrogates crash `make_doc` | `clean_text()` called before every `make_doc` |
| PDF pages joined with space | `"\n".join(...)` — preserves layout |


## 1. Install dependencies

In [1]:
!pip install -U spacy PyMuPDF scikit-learn
!python -m spacy download en_core_web_lg


     ---------------------------------------- 0.0/400.7 MB ? eta -:--:--
     ---------------------------------------- 0.0/400.7 MB ? eta -:--:--
     --------------------------------------- 1.8/400.7 MB 11.2 MB/s eta 0:00:36
     --------------------------------------- 4.7/400.7 MB 13.0 MB/s eta 0:00:31
     ---------------------------------------- 5.0/400.7 MB 8.6 MB/s eta 0:00:46
      --------------------------------------- 5.2/400.7 MB 8.2 MB/s eta 0:00:49
      --------------------------------------- 5.8/400.7 MB 5.8 MB/s eta 0:01:09
      --------------------------------------- 6.3/400.7 MB 4.9 MB/s eta 0:01:21
      --------------------------------------- 6.8/400.7 MB 4.6 MB/s eta 0:01:26
      --------------------------------------- 7.3/400.7 MB 4.3 MB/s eta 0:01:32
      --------------------------------------- 7.6/400.7 MB 4.2 MB/s eta 0:01:35
      --------------------------------------- 8.7/400.7 MB 4.0 MB/s eta 0:01:39
     - ------------------------------------- 11.0/400.

## 2. Noise filter
The dataset has ~107 annotations per CV. Many are structural words (`"skills"`, `"work"`, `"com"`,
`"marital status"`, `"is"`, `"c"`) that are labeled SKILL but confuse the model.
This filter removes them so the model learns from clean signal only.

In [2]:
NOISE_SKILLS = {
    'skills', 'work', 'knowledge', 'personal', 'professional', 'technical',
    'management', 'company', 'information', 'training', 'education',
    'qualification', 'project', 'system', 'team', 'office', 'organization',
    'experience', 'summary', 'objective', 'references', 'declaration',
    'activities', 'projects', 'about', 'profile', 'contact',
    'com', 'is', 'ltd', 'm', 'c', 'ms', 'ph', 'uc', 'xp', 'it', 'rto',
    'email', 'nationality', 'gender', 'passport',
    'marital status', 'curriculum vitae', 'work experience',
    'professional experience', 'technical skills', 'computer skills',
    'communication skills', 'interpersonal skills', 'professional qualification',
    'academic credentials', 'good communication', 'positive attitude',
    'quick learner', 'team player', 'team members', 'team work', 'hard work',
}

def is_noise(text: str) -> bool:
    t = text.strip().lower()
    if t in NOISE_SKILLS: return True
    if len(t) <= 1: return True
    if t.isdigit(): return True
    return False

print(f'Noise filter ready — blocking {len(NOISE_SKILLS)} noisy patterns.')


Noise filter ready — blocking 59 noisy patterns.


## 3. Load and clean annotations

In [3]:
import os, json

def load_spacy_data(folder_path):
    training_data = []
    skipped_noise = 0
    kept = 0
    for file_name in os.listdir(folder_path):
        if not file_name.endswith('.json'):
            continue
        with open(os.path.join(folder_path, file_name), 'r', encoding='utf-8') as f:
            data = json.load(f)
        text = data['text']
        entities = []
        for start, end, label in data['annotations']:
            if 'SKILL' not in label:
                continue
            if is_noise(text[start:end]):
                skipped_noise += 1
                continue
            entities.append((start, end, 'SKILL'))
            kept += 1
        if entities:
            training_data.append((text, {'entities': entities}))
    print(f'Documents : {len(training_data)}')
    print(f'Spans kept: {kept}   |   Filtered: {skipped_noise}')
    return training_data

FOLDER = r'C:\Users\wiame\Desktop\career-platform\ml\cv_parser\ResumesJsonAnnotated'
data = load_spacy_data(FOLDER)


Documents : 4969
Spans kept: 453523   |   Filtered: 84959


## 4. Text cleaning utility
`clean_text` **must** be called before every `make_doc` call.
Some CVs contain broken Unicode surrogates (e.g. `\ud83d` from emoji) that crash spaCy's tokenizer.

In [4]:
def clean_text(text: str) -> str:
    """Strip surrogates and null bytes without shifting character offsets."""
    if not isinstance(text, str):
        text = str(text)
    # encode+decode with errors='ignore' removes surrogates like \ud83d
    text = text.encode('utf-8', errors='ignore').decode('utf-8', errors='ignore')
    return text.replace('\x00', '')

# Quick sanity check
bad = 'hello \ud83d world'
print('Before:', repr(bad))
print('After :', repr(clean_text(bad)))


Before: 'hello \ud83d world'
After : 'hello  world'


## 5. Data health check
Run this to verify how many spans will survive conversion before committing to training.

In [5]:
import spacy
from collections import Counter

nlp_blank = spacy.blank('en')
issues = Counter()
total = 0

for text, annots in data:
    clean = clean_text(text)   # always clean before make_doc
    doc = nlp_blank.make_doc(clean)
    for start, end, _ in annots['entities']:
        total += 1
        span = doc.char_span(start, end, alignment_mode='expand')
        if span is None:
            issues['null'] += 1
        elif span.text != span.text.strip():
            issues['whitespace'] += 1

print(f'Total spans : {total}')
print(f'Issues      : {dict(issues)}')
drop = sum(issues.values())
print(f'Drop rate   : {drop/total:.1%}  (target < 3%)')


C:\Users\wiame\anaconda3\envs\cv-parser\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total spans : 453523
Issues      : {'whitespace': 7}
Drop rate   : 0.0%  (target < 3%)


## 6. Convert to spaCy format

In [6]:
from spacy.tokens import DocBin
from tqdm import tqdm

def build_docbin(data, nlp, log_path):
    db = DocBin()
    stats = Counter()
    with open(log_path, 'w', encoding='utf-8') as log:
        for text, annot in tqdm(data):
            text = clean_text(text)  # FIX: always clean before make_doc
            doc = nlp.make_doc(text)
            ents = []
            seen = set()
            for start, end, label in annot['entities']:
                while start < end and text[start].isspace(): start += 1
                while end > start and text[end-1].isspace(): end -= 1
                if start >= end:
                    stats['empty'] += 1; continue
                idx = set(range(start, end))
                if idx & seen:
                    stats['overlap'] += 1; continue
                seen.update(idx)
                span = doc.char_span(start, end, label=label, alignment_mode='expand')
                if span is None:
                    log.write(f'[NULL] ({start},{end}) {repr(text[max(0,start-15):end+15])}\n')
                    stats['null'] += 1; continue
                ents.append(span)
                stats['ok'] += 1
            try:
                doc.ents = ents
                db.add(doc)
            except Exception as e:
                log.write(f'[DOC ERROR] {e}\n')
                stats['doc_error'] += 1
    print('Span stats:', dict(stats))
    return db


## 7. Train / test split and save

In [7]:
from sklearn.model_selection import train_test_split
from pathlib import Path

train_data, test_data = train_test_split(data, test_size=0.15, random_state=42)
print(f'Train: {len(train_data)}  |  Test: {len(test_data)}')

BASE = Path(r'C:\Users\wiame\Desktop\career-platform\ml\cv_parser\model')
BASE.mkdir(parents=True, exist_ok=True)

nlp_lg = spacy.load('en_core_web_lg', exclude=['ner','tagger','parser','senter','lemmatizer'])

print('Converting train...')
train_db = build_docbin(train_data, nlp_lg, BASE / 'train_log.txt')
train_db.to_disk(BASE / 'train_data.spacy')
print('Saved train_data.spacy')

print('Converting test...')
test_db = build_docbin(test_data, nlp_lg, BASE / 'test_log.txt')
test_db.to_disk(BASE / 'test_data.spacy')
print('Saved test_data.spacy')


Train: 4223  |  Test: 746
Converting train...


100%|██████████| 4223/4223 [01:33<00:00, 45.27it/s]


Span stats: {'ok': 338072, 'overlap': 47009, 'doc_error': 446}
Saved train_data.spacy
Converting test...


100%|██████████| 746/746 [00:14<00:00, 51.12it/s]


Span stats: {'ok': 60204, 'overlap': 8238, 'doc_error': 88}
Saved test_data.spacy


## 8. Write training config

In [8]:
from pathlib import Path

CONFIG_DIR = Path(r'C:\Users\wiame\Desktop\career-platform\ml\cv_parser\config')
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

cfg = '''[paths]
train = null
dev = null
vectors = null
init_tok2vec = null

[system]
gpu_allocator = null
seed = 42

[nlp]
lang = "en"
pipeline = ["tok2vec", "ner"]
batch_size = 1000

[training]
dev_corpus = "corpora.dev"
train_corpus = "corpora.train"
seed = ${system.seed}
gpu_allocator = ${system.gpu_allocator}
dropout = 0.2
accumulate_gradient = 4
patience = 5
max_epochs = 30
max_steps = 0
eval_frequency = 200
frozen_components = []

[training.optimizer]
@optimizers = "Adam.v1"
beta1 = 0.9
beta2 = 0.999
L2_is_weight_decay = true
L2 = 0.01
grad_clip = 1.0
use_averages = false
eps = 1e-08
learn_rate = 0.001

[training.batcher]
@batchers = "spacy.batch_by_words.v1"
discard_oversize = false
tolerance = 0.2

[training.batcher.size]
@schedules = "compounding.v1"
start = 100
stop = 1000
compound = 1.001

[training.logger]
@loggers = "spacy.ConsoleLogger.v1"
progress_bar = true

[training.score_weights]
ents_f = 1.0
ents_p = 0.0
ents_r = 0.0

[corpora.train]
@readers = "spacy.Corpus.v1"
path = ${paths.train}
max_length = 0
gold_preproc = false
limit = 0

[corpora.dev]
@readers = "spacy.Corpus.v1"
path = ${paths.dev}
max_length = 0
gold_preproc = false
limit = 0

[components.tok2vec]
factory = "tok2vec"

[components.tok2vec.model]
@architectures = "spacy.Tok2Vec.v2"

[components.tok2vec.model.embed]
@architectures = "spacy.MultiHashEmbed.v2"
width = ${components.tok2vec.model.encode.width}
attrs = ["NORM", "PREFIX", "SUFFIX", "SHAPE"]
rows = [5000, 2500, 2500, 2500]
include_static_vectors = true

[components.tok2vec.model.encode]
@architectures = "spacy.MaxoutWindowEncoder.v2"
width = 256
depth = 4
window_size = 1
maxout_pieces = 3

[components.ner]
factory = "ner"
moves = null
update_with_oracle_cut_size = 100

[components.ner.model]
@architectures = "spacy.TransitionBasedParser.v2"
state_type = "ner"
extra_state_tokens = false
hidden_width = 128
maxout_pieces = 3
use_upper = true
nO = null

[components.ner.model.tok2vec]
@architectures = "spacy.Tok2VecListener.v1"
width = ${components.tok2vec.model.encode.width}
upstream = "*"

[initialize]
vectors = "en_core_web_lg"
init_tok2vec = ${paths.init_tok2vec}
'''

(CONFIG_DIR / 'config.cfg').write_text(cfg, encoding='utf-8')
print('config.cfg written')


config.cfg written


## 9. Train

In [9]:
# Remove --gpu-id 0 if running on CPU only
!python -m spacy train \
    "C:/Users/wiame/Desktop/career-platform/ml/cv_parser/config/config.cfg" \
    --output "C:/Users/wiame/Desktop/career-platform/ml/cv_parser/model/output" \
    --paths.train "C:/Users/wiame/Desktop/career-platform/ml/cv_parser/model/train_data.spacy" \
    --paths.dev   "C:/Users/wiame/Desktop/career-platform/ml/cv_parser/model/test_data.spacy" \
    --gpu-id 0


[i] Saving to output directory:
C:\Users\wiame\Desktop\career-platform\ml\cv_parser\model\output
[i] Using GPU: 0


Traceback (most recent call last):
  File "C:\Users\wiame\anaconda3\envs\cv-parser\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\wiame\anaconda3\envs\cv-parser\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "C:\Users\wiame\anaconda3\envs\cv-parser\lib\site-packages\spacy\__main__.py", line 4, in <module>
    setup_cli()
  File "C:\Users\wiame\anaconda3\envs\cv-parser\lib\site-packages\spacy\cli\_util.py", line 78, in setup_cli
    command(prog_name=COMMAND)
  File "C:\Users\wiame\anaconda3\envs\cv-parser\lib\site-packages\click\core.py", line 1514, in __call__
    return self.main(*args, **kwargs)
  File "C:\Users\wiame\anaconda3\envs\cv-parser\lib\site-packages\typer\core.py", line 794, in main
    return _main(
  File "C:\Users\wiame\anaconda3\envs\cv-parser\lib\site-packages\typer\core.py", line 188, in _main
    rv = self.invoke(ctx)
  File "C:\Users\wiame\anaconda3\envs\cv-parser\lib\site-pa

## 10. Read your scores

In [10]:
import json
from pathlib import Path

meta_path = Path(r'C:\Users\wiame\Desktop\career-platform\ml\cv_parser\model\output\model-best\meta.json')
with open(meta_path) as f:
    meta = json.load(f)

skill = meta['performance']['ents_per_type']['SKILL']
print('=' * 40)
print(f'  Precision : {skill["p"]*100:.1f}%')
print(f'  Recall    : {skill["r"]*100:.1f}%')
print(f'  F1 Score  : {skill["f"]*100:.1f}%')
print('=' * 40)
if skill['f'] >= 0.80:   print('Ready for production')
elif skill['f'] >= 0.70: print('Good — acceptable for most use cases')
else:                     print('Run error analysis below')


  Precision : 63.4%
  Recall    : 80.1%
  F1 Score  : 70.8%
Good — acceptable for most use cases


## 11. Error analysis

In [11]:
import spacy
from collections import Counter

nlp = spacy.load(r'C:\Users\wiame\Desktop\career-platform\ml\cv_parser\model\output\model-best')

fn, fp = [], []
for text, annots in test_data[:100]:
    gold = set(text[s:e].strip().lower() for s, e, _ in annots['entities'])
    pred = set(ent.text.strip().lower() for ent in nlp(text).ents)
    fn += list(gold - pred)
    fp += list(pred - gold)

print('Top 20 MISSED skills (add more annotations for these):')
for s, c in Counter(fn).most_common(20): print(f'  {c:3d}x  {s}')
print()
print('Top 20 HALLUCINATED skills (false positives):')
for s, c in Counter(fp).most_common(20): print(f'  {c:3d}x  {s}')


Top 20 MISSED skills (add more annotations for these):
   54x  gmail
   25x  engineering
   24x  data
   23x  engineer
   21x  mechanical
   21x  ms office
   20x  business
   20x  communication
   20x  support
   20x  power
   17x  systems
   16x  microsoft office
   16x  manager
   16x  quality
   15x  tech
   15x  maintenance
   14x  construction
   14x  curriculum
   14x  software
   14x  design

Top 20 HALLUCINATED skills (false positives):
   67x  personal
   51x  skills
   50x  office
   48x  team
   48x  information
   46x  education
   46x  knowledge
   44x  nationality
   41x  professional
   40x  activities
   40x  is
   39x  company
   36x  work
   33x  system
   32x  organization
   31x  marital status
   30x  email
   30x  ms
   30x  qualification
   29x  management


## 12. Test on real PDF CVs

In [13]:
import fitz, os

def extract_skills_from_pdf(pdf_path, nlp):
    pdf = fitz.open(pdf_path)
    text = '\n'.join(page.get_text() for page in pdf)  # newline not space
    doc = nlp(text)
    return sorted(set(
        ent.text.strip() for ent in doc.ents
        if ent.label_ == 'SKILL' and len(ent.text.strip()) > 1
    ))

TEST_DIR = r'C:\Users\wiame\Desktop\career-platform\ml\cv_parser\test'
for fname in os.listdir(TEST_DIR):
    if not fname.endswith('.pdf'): continue
    skills = extract_skills_from_pdf(os.path.join(TEST_DIR, fname), nlp)
    print(f'\n{fname}  ({len(skills)} skills)')
    for s in skills: print(f'  * {s}')



CarterAndradeResume.pdf  (80 skills)
  * 10+ years of experience
  * AWS
  * Azure
  * Big data
  * Cloud
  * Cloud Security
  * College
  * Communication Skills
  * Computer Science
  * Creativity
  * Data Analysis
  * Data Engineer
  * EDUCATION
  * EMR
  * ETL
  * Email
  * Epic
  * Hadoop
  * Hive
  * Kafka
  * Leading
  * MongoDB
  * MySQL
  * Oracle
  * Pipelines
  * Programming
  * Protection
  * Python
  * SKILLS
  * SQL
  * SQL Server
  * STRENGTHS
  * San
  * Scala
  * Senior
  * Spark
  * TX
  * Talend
  * Talent
  * accuracy
  * acquisition
  * biotechnology
  * building
  * challenging
  * company
  * compatibility
  * data
  * data analysis
  * data pipeline
  * data structures
  * development
  * efficiency
  * effort
  * engineers
  * focused
  * healthcare
  * implementing
  * indexing
  * is
  * it
  * legacy
  * load
  * make
  * optimization
  * problems
  * process
  * processing
  * query
  * researching
  * scalability
  * sequence
  * solutions
  * sorting
  * 

## 13. Web app helper (drop into FastAPI / Flask)

In [ ]:
import spacy, fitz

_cache = {}
def get_model(path): 
    if path not in _cache: _cache[path] = spacy.load(path)
    return _cache[path]

def extract_skills_from_text(text: str, model_path: str) -> list:
    nlp = get_model(model_path)
    doc = nlp(text)
    return sorted(set(
        ent.text.strip() for ent in doc.ents
        if ent.label_ == 'SKILL' and len(ent.text.strip()) > 1
    ))

def extract_skills_from_pdf(pdf_path: str, model_path: str) -> dict:
    pdf = fitz.open(pdf_path)
    text = '\n'.join(page.get_text() for page in pdf)
    skills = extract_skills_from_text(text, model_path)
    return {'skills': skills, 'skill_count': len(skills), 'pages': len(pdf)}

# Smoke test
MODEL = r'C:\Users\wiame\Desktop\career-platform\ml\cv_parser\model\output\model-best'
PDF   = r'C:\Users\wiame\Desktop\career-platform\ml\cv_parser\test\CV_Erraoui_Wiame.pdf'
import json; print(json.dumps(extract_skills_from_pdf(PDF, MODEL), indent=2))


In [14]:
import shutil
from pathlib import Path

# ── Change this to wherever you want to save the model ──────────────────────
SAVE_DIR = Path(r"C:\Users\wiame\Desktop\career-platform\ml\cv_parser\model\saved_model")

# Copy model-best to your chosen location
MODEL_BEST = Path(r"C:\Users\wiame\Desktop\career-platform\ml\cv_parser\model\output\model-best")
shutil.copytree(MODEL_BEST, SAVE_DIR, dirs_exist_ok=True)

print(f"✔ Model saved to: {SAVE_DIR}")
print(f"  Files saved: {[f.name for f in SAVE_DIR.iterdir()]}")

✔ Model saved to: C:\Users\wiame\Desktop\career-platform\ml\cv_parser\model\saved_model
  Files saved: ['config.cfg', 'meta.json', 'ner', 'tok2vec', 'tokenizer', 'vocab']
